In [ ]:
!pip -q install -U transformers sentence-transformers rank_bm25 accelerate

In [ ]:
import json, glob, re, random, unicodedata, numpy as np, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

chunks = [json.loads(l) for l in open(find("chunks.jsonl"), encoding="utf-8")]
qa = [json.loads(l) for l in open(find("qa.jsonl"), encoding="utf-8")]
qa = [r for r in qa if r["kb_chunk_ids"]]
ids = [c["chunk_id"] for c in chunks]
N = len(ids)
fact = lambda r: r["notes"].split(";")[0]
facts = sorted({fact(r) for r in qa})
random.Random(42).shuffle(facts)
dev_facts = set(facts[: len(facts) // 2])
latn = [r for r in qa if r["lang"] == "ur-Latn"]
test = [r for r in latn if fact(r) not in dev_facts]

# 1. Sarvam-Translate: Roman Urdu -> Urdu script
name = "sarvamai/sarvam-translate"
tokzr = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto")
def convert(text):
    msgs = [{"role": "system", "content": "Translate the text below to Urdu."}, {"role": "user", "content": text}]
    x = tokzr.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**x, max_new_tokens=80, do_sample=False)
    return tokzr.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip().splitlines()[0].strip()
URDU = re.compile(r"^[؀-ۿݐ-ݿﭐ-﷿ﹰ-\uFEFF0-9\s\-،؟?!.,:()/%+]+$")
translit = {r["id"]: convert(r["question"]) for r in latn}
bad = [k for k, v in translit.items() if not URDU.match(v)]
print(f"conversions: {len(translit)}, not pure Urdu: {len(bad)} (Qwen 14B was 16)")
for k in list(translit)[:8]:
    print(k, translit[k])
del llm; torch.cuda.empty_cache()

# 2. Retrieval: RRF + reranker on the unseen half
FOLD = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ە": "ہ", "ۀ": "ۂ"})
tok = lambda s: re.findall(r"\w+", unicodedata.normalize("NFKC", s).translate(FOLD).lower())
bm = BM25Okapi([tok(c["text"]) for c in chunks])
enc = SentenceTransformer("BAAI/bge-m3", device="cuda"); enc.max_seq_length = 1024
rer = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=1024)
C = enc.encode([c["text"] for c in chunks], batch_size=8, normalize_embeddings=True)
def rank(q):
    f = np.zeros(N)
    for s in (bm.get_scores(tok(q)), C @ enc.encode(q, normalize_embeddings=True)):
        for p, j in enumerate(np.argsort(-s)):
            f[j] += 1 / (60 + p + 1)
    order = [int(j) for j in np.argsort(-f)]
    sc = rer.predict([(q, chunks[j]["text"]) for j in order[:20]], batch_size=8)
    return [order[i] for i in np.argsort(-sc)] + order[20:]
r1, r5, mrr = [], [], []
for r in test:
    rk, gold = rank(translit[r["id"]]), set(r["kb_chunk_ids"])
    r1.append(ids[rk[0]] in gold)
    r5.append(any(ids[j] in gold for j in rk[:5]))
    mrr.append(next((1 / (p + 1) for p, j in enumerate(rk) if ids[j] in gold), 0.0))
res = {"r1": round(np.mean(r1), 3), "r5": round(np.mean(r5), 3), "mrr": round(np.mean(mrr), 3)}
print(f"\nur-Latn unseen half (n={len(test)}), Sarvam conversion: {res}  (Qwen 14B: r1 0.439, r5 0.902, mrr 0.624)")

json.dump({"converter": name, "not_pure_urdu": len(bad), "results": res, "transliterations": translit},
          open("/kaggle/working/e3_sarvam_kb_v2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("saved /kaggle/working/e3_sarvam_kb_v2.json")
